# Production Prompt Lifecycle

This notebook teaches you how to manage the **full prompt lifecycle** in production using [Langfuse](https://langfuse.com/) and AWS services. You will learn to version prompts, run systematic evaluations, automate quality scoring, and integrate everything into a CI/CD pipeline.

## Learning Objectives

By the end of this notebook, you will be able to:
- **Version and manage prompts** in Langfuse (not just Git)
- **Create systematic evaluation datasets** for repeatable testing
- **Implement automated quality scoring** with LLM-as-Judge
- **Integrate Langfuse with CI/CD pipelines** using AWS CodePipeline

## Why This Matters

Git tracks code changes, but it cannot tell you which prompt version produced which quality metrics in production. Langfuse bridges this gap by linking prompt versions to traces, costs, and latency data -- giving you a complete picture of how each prompt performs.

| Challenge | Git Alone | Git + Langfuse |
|-----------|-----------|----------------|
| Version tracking | File diffs | Prompt versions linked to traces |
| Performance analysis | Manual log review | Cost, latency, token usage per version |
| A/B testing | Feature flags | Label-based routing with analytics |
| Quality gates | Unit tests only | LLM-as-Judge + dataset experiments |

**Duration**: ~75-90 minutes

## Prerequisites

Before running this notebook, ensure you have:
1. Completed the [02-developer-journey](../02-developer-journey/) notebooks (Langfuse already configured)
2. A self-hosted or cloud Langfuse instance
3. AWS credentials with Bedrock access

## Setup

In [ ]:
from __future__ import annotations

from dotenv import load_dotenv

load_dotenv()

import json
import os
import time
from pathlib import Path

import boto3

# ── AWS Bedrock client ──────────────────────────────────────────────────
REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

# Model configuration
MODEL_SONNET = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"
MODEL_HAIKU = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_ID = MODEL_SONNET        # Primary model for agent responses
JUDGE_MODEL_ID = MODEL_HAIKU   # Cheaper model for LLM-as-Judge

# ── Langfuse client (graceful degradation if not configured) ──────────
LANGFUSE_AVAILABLE = False
langfuse = None

try:
    from langfuse import Langfuse

    public_key = os.environ.get("LANGFUSE_PUBLIC_KEY")
    secret_key = os.environ.get("LANGFUSE_SECRET_KEY")
    langfuse_host = os.environ.get("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")

    if public_key and secret_key:
        langfuse = Langfuse(
            public_key=public_key,
            secret_key=secret_key,
            host=langfuse_host,
        )
        LANGFUSE_AVAILABLE = True
        print(f"Langfuse connected: {langfuse_host}")
    else:
        print(
            "WARNING: LANGFUSE_PUBLIC_KEY or LANGFUSE_SECRET_KEY not set.\n"
            "Langfuse features will be simulated with print statements."
        )
except ImportError:
    print(
        "WARNING: langfuse package not installed.\n"
        "Run: pip install langfuse\n"
        "Langfuse features will be simulated with print statements."
    )

# ── Helper: call Bedrock Converse API ─────────────────────────────────


def call_bedrock(
    prompt: str,
    system: str | None = None,
    temperature: float = 0.0,
    max_tokens: int = 500,
    model_id: str | None = None,
) -> dict:
    """Call Bedrock Converse API and return text + usage metrics."""
    model = model_id or MODEL_ID
    messages = [{"role": "user", "content": [{"text": prompt}]}]

    kwargs: dict = {
        "modelId": model,
        "messages": messages,
        "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature},
    }
    if system:
        kwargs["system"] = [{"text": system}]

    response = bedrock_runtime.converse(**kwargs)
    usage = response["usage"]

    return {
        "text": response["output"]["message"]["content"][0]["text"],
        "input_tokens": usage["inputTokens"],
        "output_tokens": usage["outputTokens"],
        "latency_ms": response["metrics"]["latencyMs"],
    }


print(f"Region:       {REGION}")
print(f"Agent Model:  {MODEL_ID}")
print(f"Judge Model:  {JUDGE_MODEL_ID}")
print(f"Langfuse:     {'connected' if LANGFUSE_AVAILABLE else 'not configured (simulated mode)'}")
print("\nSetup complete!")

<div class="alert alert-block alert-info">
<b>Note:</b> If Langfuse credentials are not configured, this notebook will run in <b>simulated mode</b> -- all Langfuse operations are replaced with local print statements so you can still follow along and understand the concepts.
</div>

---

# Section 1: Langfuse Prompt Management (20 min)

Langfuse Prompt Management lets you store prompt versions alongside their performance data. Unlike Git, which only tracks text changes, Langfuse links each prompt version to the traces it produced -- giving you cost, latency, and quality metrics per version.

| Concept | Description |
|---------|-------------|
| **Why Langfuse vs Git** | Git versions code; Langfuse links prompts to traces for performance analysis |
| **Production labels** | Mark versions as `production` for safe, label-based deployments |
| **Prompt-trace linkage** | Every trace records which prompt version was used |

**References:**
- [Langfuse Prompt Management](https://langfuse.com/docs/prompts/get-started)
- [Linking Prompts to Traces](https://langfuse.com/docs/prompts/get-started#link-prompts-to-traces)

## Demo 1: Create Prompt Versions in Langfuse

We will create two versions of a customer support system prompt:

- **v1 (baseline)**: Verbose, detailed instructions
- **v2 (optimized)**: Concise, production-ready

Calling `create_prompt` with the same `name` automatically creates a new version.

In [ ]:
# ── Prompt definitions ─────────────────────────────────────────────────

PROMPT_NAME = "customer-support-system"

PROMPT_V1_TEXT = """You are a helpful customer support assistant for TechMart Electronics.

Your role is to assist customers with:
- Product information and specifications
- Return policies and procedures
- Technical troubleshooting
- Order status inquiries

Always be polite, professional, and thorough in your responses.
If you don't know the answer, say so and offer to escalate to a human agent.
Provide detailed explanations and walk the customer through each step."""

PROMPT_V2_TEXT = (
    "TechMart Electronics support assistant. "
    "Help with: products, returns, tech support, orders. "
    "Be concise and professional. Escalate unknowns."
)

# Model configuration stored alongside the prompt
CONFIG_V1 = {
    "model": "anthropic.claude-sonnet-4-5-20250929-v1:0",
    "temperature": 0.0,
    "max_tokens": 1000,
}

CONFIG_V2 = {
    "model": "anthropic.claude-sonnet-4-5-20250929-v1:0",
    "temperature": 0.0,
    "max_tokens": 500,
}

# ── Create prompt versions ─────────────────────────────────────────────

if LANGFUSE_AVAILABLE:
    # v1: Draft (no production label)
    langfuse.create_prompt(
        name=PROMPT_NAME,
        type="text",
        prompt=PROMPT_V1_TEXT,
        config=CONFIG_V1,
        labels=[],  # Draft -- not yet promoted
    )
    print("Created prompt v1 (draft)")

    # v2: Labeled as production
    langfuse.create_prompt(
        name=PROMPT_NAME,
        type="text",
        prompt=PROMPT_V2_TEXT,
        config=CONFIG_V2,
        labels=["production"],  # Promoted to production
    )
    print("Created prompt v2 (production)")
    print(f"\nView prompts: {langfuse_host}/prompts")
else:
    print("[Simulated] Would create two prompt versions in Langfuse:")
    print(f"  v1 (draft):      {len(PROMPT_V1_TEXT)} chars")
    print(f"  v2 (production): {len(PROMPT_V2_TEXT)} chars")

## Demo 2: Fetch Prompt at Runtime

In production, your agent fetches the `production`-labeled prompt at startup. This decouples prompt changes from code deployments -- you can update prompts in Langfuse without redeploying the application.

In [ ]:
# ── Fetch production prompt and use it ─────────────────────────────────

if LANGFUSE_AVAILABLE:
    # Fetch the prompt labeled "production" (v2 in our case)
    prompt_obj = langfuse.get_prompt(PROMPT_NAME, label="production")

    # .compile() returns the prompt text (with variable substitution if any)
    system_prompt = prompt_obj.compile()
    model_config = prompt_obj.config

    print(f"Fetched prompt: {PROMPT_NAME}")
    print(f"  Version:     {prompt_obj.version}")
    print(f"  Labels:      {prompt_obj.labels}")
    print(f"  Config:      {json.dumps(model_config, indent=2)}")
    print(f"  Text:        {system_prompt[:80]}...")
else:
    # Fallback: use the v2 prompt directly
    system_prompt = PROMPT_V2_TEXT
    model_config = CONFIG_V2
    prompt_obj = None
    print("[Simulated] Using v2 prompt text directly")
    print(f"  Text: {system_prompt[:80]}...")

# ── Call agent with the fetched prompt ──────────────────────────────────

user_message = "What is your return policy for laptops?"

# Create a trace linked to the prompt version
trace = None
if LANGFUSE_AVAILABLE:
    trace = langfuse.trace(name="customer-support-demo")
    generation = trace.generation(
        name="llm-call",
        prompt=prompt_obj,  # Links this trace to the prompt version
        input=user_message,
    )

response = call_bedrock(
    prompt=user_message,
    system=system_prompt,
    temperature=model_config.get("temperature", 0.0),
    max_tokens=model_config.get("max_tokens", 500),
)

if LANGFUSE_AVAILABLE and trace is not None:
    generation.end(output=response["text"])
    trace.update(output=response["text"])
    langfuse.flush()

print(f"\nQuery: {user_message}")
print(f"Response: {response['text'][:300]}...")
print(f"\nTokens: input={response['input_tokens']}, output={response['output_tokens']}")
print(f"Latency: {response['latency_ms']}ms")

## Demo 3: Analyze Performance by Prompt Version

Let us run both prompt versions against the same queries and compare their token usage and latency. In production, you would view this comparison in the Langfuse dashboard.

In [ ]:
# ── Compare v1 vs v2 performance ──────────────────────────────────────

test_queries = [
    "What is your return policy for laptops?",
    "My laptop won't turn on, can you help?",
    "What smartphones do you carry?",
]

print("=" * 70)
print("PROMPT VERSION COMPARISON: v1 (verbose) vs v2 (concise)")
print("=" * 70)

versions = {
    "v1-verbose": {"prompt": PROMPT_V1_TEXT, "config": CONFIG_V1},
    "v2-concise": {"prompt": PROMPT_V2_TEXT, "config": CONFIG_V2},
}

results_by_version: dict[str, list[dict]] = {}

for version_name, version_data in versions.items():
    print(f"\n--- {version_name} ---")
    version_results = []

    for query in test_queries:
        resp = call_bedrock(
            prompt=query,
            system=version_data["prompt"],
            temperature=version_data["config"]["temperature"],
            max_tokens=version_data["config"]["max_tokens"],
        )
        version_results.append(resp)
        print(
            f"  [{resp['input_tokens']:>4} in, {resp['output_tokens']:>4} out, "
            f"{resp['latency_ms']:>5}ms] {query[:50]}"
        )

    results_by_version[version_name] = version_results

# ── Summary comparison ────────────────────────────────────────────────
print(f"\n{'=' * 70}")
print(f"{'Version':<15} {'Avg Input':>12} {'Avg Output':>12} {'Avg Latency':>14}")
print("-" * 70)

for version_name, results_list in results_by_version.items():
    avg_in = sum(r["input_tokens"] for r in results_list) / len(results_list)
    avg_out = sum(r["output_tokens"] for r in results_list) / len(results_list)
    avg_lat = sum(r["latency_ms"] for r in results_list) / len(results_list)
    print(f"{version_name:<15} {avg_in:>10.0f} tk {avg_out:>10.0f} tk {avg_lat:>11.0f} ms")

print("=" * 70)
print("\nThe concise prompt typically uses fewer input tokens while")
print("maintaining comparable output quality.")

if LANGFUSE_AVAILABLE:
    print(f"\nView per-version analytics in Langfuse: {langfuse_host}/prompts")

---

# Section 2: Systematic Evaluation with Datasets (20 min)

Langfuse Datasets let you store test cases centrally and run repeatable experiments across prompt versions. Unlike scattered JSON files, datasets in Langfuse are linked to experiments so you can compare results side-by-side.

| Concept | Description |
|---------|-------------|
| **Datasets as source of truth** | Test cases stored in Langfuse, not scattered JSON files |
| **Experiments** | Run same dataset against different versions, compare results |
| **Dataset items from production** | Convert real failures into test cases |

**References:**
- [Langfuse Datasets Overview](https://langfuse.com/docs/datasets/overview)
- [Running Experiments](https://langfuse.com/docs/datasets/python-cookbook)

## Demo 1: Create Evaluation Dataset in Langfuse

We will load test cases from a local JSON file and upload them to Langfuse as a dataset. Each item has:
- **input**: The customer query
- **expected_output**: Keywords the response should contain, expected tool usage
- **metadata**: Category, complexity level

In [ ]:
# ── Load local evaluation dataset ─────────────────────────────────────

EVAL_DATASET_PATH = Path("data/customer-support-eval.json")
DATASET_NAME = "customer-support-eval"

with open(EVAL_DATASET_PATH, encoding="utf-8") as f:
    eval_test_cases = json.load(f)

print(f"Loaded {len(eval_test_cases)} test cases from {EVAL_DATASET_PATH}")
print("\nSample entries:")
for case in eval_test_cases[:3]:
    print(f"  [{case['metadata']['category']:>14}] {case['input']['query'][:55]}...")

# ── Upload to Langfuse as a dataset ────────────────────────────────────

if LANGFUSE_AVAILABLE:
    # Create (or update) the dataset
    langfuse.create_dataset(
        name=DATASET_NAME,
        description="Core evaluation scenarios for customer support agent",
        metadata={"version": "1.0", "created_by": "workshop"},
    )

    for case in eval_test_cases:
        langfuse.create_dataset_item(
            dataset_name=DATASET_NAME,
            input=case["input"],
            expected_output=case.get("expected_output", {}),
            metadata=case.get("metadata", {}),
        )

    langfuse.flush()
    print(f"\nUploaded {len(eval_test_cases)} items to Langfuse dataset: {DATASET_NAME}")
    print(f"View dataset: {langfuse_host}/datasets")
else:
    print(f"\n[Simulated] Would upload {len(eval_test_cases)} items to Langfuse dataset: {DATASET_NAME}")

## Demo 2: Run Experiments Against Prompt Versions

An **experiment** runs every dataset item through a specific agent configuration and records the results. We will run two experiments:
1. `v1-baseline` -- using the verbose prompt
2. `v2-optimized` -- using the concise prompt

Each experiment result is linked back to the dataset item, enabling side-by-side comparison.

In [ ]:
# ── Run experiment helper ─────────────────────────────────────────────


def run_experiment(
    system_prompt_text: str,
    experiment_name: str,
    test_cases: list[dict],
    max_tokens: int = 500,
) -> list[dict]:
    """Run all test cases through the agent and record results.

    If Langfuse is available, traces are created and linked to dataset items.
    Returns a list of result dicts with query, response, and token metrics.
    """
    experiment_results = []

    # Fetch dataset items from Langfuse if available
    langfuse_dataset = None
    if LANGFUSE_AVAILABLE:
        try:
            langfuse_dataset = langfuse.get_dataset(DATASET_NAME)
        except Exception:
            pass  # Fall back to local test cases

    for idx, case in enumerate(test_cases):
        query = case["input"]["query"]
        expected = case.get("expected_output", {})

        # Create Langfuse trace
        trace = None
        if LANGFUSE_AVAILABLE:
            trace = langfuse.trace(
                name=f"eval-{experiment_name}",
                metadata={"experiment": experiment_name, "item_idx": idx},
            )

        # Call agent
        resp = call_bedrock(
            prompt=query,
            system=system_prompt_text,
            max_tokens=max_tokens,
        )

        # Link trace to dataset item
        if LANGFUSE_AVAILABLE and trace is not None:
            trace.update(output=resp["text"])

            # Link to the corresponding Langfuse dataset item
            if langfuse_dataset and idx < len(langfuse_dataset.items):
                langfuse_dataset.items[idx].link(
                    trace=trace,
                    run_name=experiment_name,
                    run_metadata={"agent_version": experiment_name},
                )

        experiment_results.append({
            "query": query,
            "response": resp["text"],
            "input_tokens": resp["input_tokens"],
            "output_tokens": resp["output_tokens"],
            "latency_ms": resp["latency_ms"],
            "expected": expected,
        })

    if LANGFUSE_AVAILABLE:
        langfuse.flush()

    return experiment_results


# ── Run experiments ───────────────────────────────────────────────────

print("Running experiment: v1-baseline")
exp_v1 = run_experiment(
    system_prompt_text=PROMPT_V1_TEXT,
    experiment_name="v1-baseline",
    test_cases=eval_test_cases,
    max_tokens=CONFIG_V1["max_tokens"],
)
print(f"  Completed {len(exp_v1)} items")

print("\nRunning experiment: v2-optimized")
exp_v2 = run_experiment(
    system_prompt_text=PROMPT_V2_TEXT,
    experiment_name="v2-optimized",
    test_cases=eval_test_cases,
    max_tokens=CONFIG_V2["max_tokens"],
)
print(f"  Completed {len(exp_v2)} items")

## Demo 3: Compare Experiment Results

In the Langfuse UI, navigate to **Datasets > customer-support-eval > Runs** to see a side-by-side comparison. Below we show the comparison locally.

In [ ]:
# ── Compare experiments locally ───────────────────────────────────────

print("=" * 70)
print("EXPERIMENT COMPARISON: v1-baseline vs v2-optimized")
print("=" * 70)


def summarize_experiment(name: str, results: list[dict]) -> dict:
    """Compute summary statistics for an experiment."""
    total_input = sum(r["input_tokens"] for r in results)
    total_output = sum(r["output_tokens"] for r in results)
    avg_latency = sum(r["latency_ms"] for r in results) / len(results)
    return {
        "name": name,
        "items": len(results),
        "total_input": total_input,
        "total_output": total_output,
        "avg_input": total_input / len(results),
        "avg_output": total_output / len(results),
        "avg_latency_ms": avg_latency,
    }


summary_v1 = summarize_experiment("v1-baseline", exp_v1)
summary_v2 = summarize_experiment("v2-optimized", exp_v2)

print(f"\n{'Metric':<22} {'v1-baseline':>14} {'v2-optimized':>14} {'Change':>10}")
print("-" * 70)

for metric, key in [
    ("Avg Input Tokens", "avg_input"),
    ("Avg Output Tokens", "avg_output"),
    ("Avg Latency (ms)", "avg_latency_ms"),
    ("Total Input Tokens", "total_input"),
    ("Total Output Tokens", "total_output"),
]:
    v1_val = summary_v1[key]
    v2_val = summary_v2[key]
    if v1_val > 0:
        change = ((v2_val - v1_val) / v1_val) * 100
        change_str = f"{change:+.1f}%"
    else:
        change_str = "N/A"
    print(f"{metric:<22} {v1_val:>14,.0f} {v2_val:>14,.0f} {change_str:>10}")

print("=" * 70)

if LANGFUSE_AVAILABLE:
    print(f"\nView experiment runs side-by-side: {langfuse_host}/datasets")

---

# Section 3: Automated Quality Scoring (15 min)

Numbers alone (tokens, latency) do not tell you if the agent's answers are actually good. Langfuse Scores let you attach quality ratings to traces -- either manually (human review) or automatically (LLM-as-Judge).

| Concept | Description |
|---------|-------------|
| **Scores** | Numeric, categorical, or boolean quality ratings attached to traces |
| **Manual vs Automated** | Human review vs LLM-as-Judge |
| **Score analytics** | Track quality trends, set alerts on regressions |

**References:**
- [Langfuse Scores Overview](https://langfuse.com/docs/scores/overview)
- [Model-Based Evals (LLM-as-Judge)](https://langfuse.com/docs/scores/model-based-evals)
- [External Evaluation Pipelines](https://langfuse.com/docs/scores/external-evaluation-pipelines)

## Demo 1: Keyword-Based Scoring

The simplest scoring approach: check whether the response contains expected keywords from the evaluation dataset.

In [ ]:
# ── Keyword-based scoring ─────────────────────────────────────────────


def score_keyword_accuracy(response_text: str, expected: dict) -> float:
    """Score based on whether expected keywords appear in the response.

    Returns a score between 0.0 and 1.0.
    """
    score = 0.0
    checks = 0

    if "should_contain" in expected:
        for keyword in expected["should_contain"]:
            checks += 1
            if keyword.lower() in response_text.lower():
                score += 1

    if "should_use_tool" in expected:
        checks += 1
        # In a real system, check tool invocation logs.
        # Here we check if the tool concept is mentioned.
        if expected["should_use_tool"].replace("_", " ").lower() in response_text.lower():
            score += 1

    return score / checks if checks > 0 else 1.0


# ── Score v2 experiment results ────────────────────────────────────────

print("=" * 70)
print("KEYWORD ACCURACY SCORES (v2-optimized)")
print("=" * 70)

keyword_scores = []
for result in exp_v2:
    kw_score = score_keyword_accuracy(result["response"], result["expected"])
    keyword_scores.append(kw_score)
    status = "PASS" if kw_score >= 0.5 else "FAIL"
    print(f"  [{status}] {kw_score:.2f}  {result['query'][:55]}")

avg_keyword = sum(keyword_scores) / len(keyword_scores)
print(f"\nAverage keyword accuracy: {avg_keyword:.3f}")
print(f"Pass rate (>= 0.5):      {sum(1 for s in keyword_scores if s >= 0.5) / len(keyword_scores):.1%}")

## Demo 2: LLM-as-Judge Evaluator

Keyword matching is brittle -- the agent might provide a correct answer using different wording. An **LLM-as-Judge** evaluator uses a cheaper model (Haiku) to assess response quality on a 0-10 scale.

<div class="alert alert-block alert-info">
<b>Cost tip:</b> We use <code>Claude Haiku 4.5</code> as the judge model. At ~12x cheaper than Sonnet for input tokens, running evaluations on 12 test cases costs less than $0.01.
</div>

In [ ]:
# ── LLM-as-Judge evaluator ────────────────────────────────────────────


def evaluate_helpfulness(query: str, response_text: str) -> dict:
    """Use Claude Haiku to score helpfulness on a 0-10 scale.

    Returns a dict with 'score' (normalized 0-1) and 'reason'.
    """
    eval_prompt = f"""Rate the helpfulness of this customer support response on a scale of 0 to 10.

Customer Query: {query}

Agent Response: {response_text}

Scoring criteria:
- 0-3: Unhelpful, incorrect, or confusing
- 4-6: Partially helpful, missing key information
- 7-9: Helpful, accurate, addresses the query well
- 10: Exceptional, exceeds expectations

Return ONLY a JSON object with this exact format:
{{"score": <number>, "reason": "<one sentence explanation>"}}"""

    try:
        resp = call_bedrock(
            prompt=eval_prompt,
            model_id=JUDGE_MODEL_ID,
            max_tokens=100,
            temperature=0.0,
        )
        parsed = json.loads(resp["text"].strip())
        return {
            "score": float(parsed.get("score", 0)) / 10.0,
            "reason": parsed.get("reason", ""),
        }
    except (json.JSONDecodeError, KeyError, ValueError):
        return {"score": 0.0, "reason": "Failed to parse judge response"}


# ── Score all v2 experiment results ────────────────────────────────────

print("=" * 70)
print("LLM-AS-JUDGE HELPFULNESS SCORES (v2-optimized)")
print("=" * 70)

judge_scores = []
for result in exp_v2:
    judge_result = evaluate_helpfulness(result["query"], result["response"])
    judge_scores.append(judge_result)
    print(
        f"  {judge_result['score']:.2f}  {result['query'][:45]:<45}  "
        f"{judge_result['reason'][:50]}"
    )

avg_judge = sum(s["score"] for s in judge_scores) / len(judge_scores)
print(f"\nAverage helpfulness score: {avg_judge:.3f}")
print(f"Items scoring >= 0.7:     {sum(1 for s in judge_scores if s['score'] >= 0.7)}/{len(judge_scores)}")

## Demo 3: Push Scores to Langfuse and View Analytics

Now we push both keyword and LLM-as-Judge scores back to Langfuse, attached to the traces we created during the experiment. In the Langfuse UI, navigate to **Scores** to see trends over time.

In [ ]:
# ── Push scores to Langfuse ───────────────────────────────────────────

if LANGFUSE_AVAILABLE:
    # Fetch traces from the v2-optimized experiment
    try:
        traces_response = langfuse.api.trace.list(limit=len(exp_v2))
        recent_traces = traces_response.data if traces_response else []

        scored_count = 0
        for i, trace_obj in enumerate(recent_traces[: len(exp_v2)]):
            if i < len(keyword_scores):
                langfuse.create_score(
                    trace_id=trace_obj.id,
                    name="keyword_accuracy",
                    value=keyword_scores[i],
                    comment=f"Keyword match score: {keyword_scores[i]:.2f}",
                )
            if i < len(judge_scores):
                langfuse.create_score(
                    trace_id=trace_obj.id,
                    name="helpfulness_llm",
                    value=judge_scores[i]["score"],
                    comment=judge_scores[i]["reason"],
                )
            scored_count += 1

        langfuse.flush()
        print(f"Pushed scores for {scored_count} traces to Langfuse")
        print(f"View score analytics: {langfuse_host}/scores")
    except Exception as e:
        print(f"Could not push scores to Langfuse: {e}")
        print("Scores computed locally (see above).")
else:
    print("[Simulated] Scores computed locally (see above).")
    print("With Langfuse connected, scores would appear in the Scores dashboard.")

# ── Local summary ─────────────────────────────────────────────────────

print(f"\n{'=' * 70}")
print("COMBINED SCORE SUMMARY (v2-optimized)")
print(f"{'=' * 70}")

combined_scores = []
for i in range(len(exp_v2)):
    kw = keyword_scores[i] if i < len(keyword_scores) else 0.0
    jd = judge_scores[i]["score"] if i < len(judge_scores) else 0.0
    combined = (kw + jd) / 2.0
    combined_scores.append(combined)

avg_combined = sum(combined_scores) / len(combined_scores)
pass_count = sum(1 for s in combined_scores if s >= 0.5)

print(f"  Average keyword score:      {avg_keyword:.3f}")
print(f"  Average judge score:        {avg_judge:.3f}")
print(f"  Average combined score:     {avg_combined:.3f}")
print(f"  Pass rate (combined >= 0.5): {pass_count}/{len(combined_scores)} ({pass_count / len(combined_scores):.1%})")
print(f"{'=' * 70}")

# Quality gate check
SCORE_THRESHOLD = 0.7
if avg_combined >= SCORE_THRESHOLD:
    print(f"\nQuality gate: PASSED (avg {avg_combined:.3f} >= {SCORE_THRESHOLD})")
else:
    print(f"\nQuality gate: FAILED (avg {avg_combined:.3f} < {SCORE_THRESHOLD})")
    print("Action: Review low-scoring items and iterate on the prompt.")

---

# Section 4: CI/CD Integration with AWS CodePipeline (20 min)

In production, prompt changes should go through the same rigor as code changes: automated testing, quality gates, and approval workflows. This section shows how to integrate Langfuse evaluations into an AWS CodePipeline.

| Concept | Description |
|---------|-------------|
| **Langfuse as quality gate** | CI/CD checks Langfuse scores before deployment |
| **External evaluation pipeline** | Script that fetches dataset, runs evals, pushes scores |
| **CodePipeline + Lambda** | AWS-native CI/CD with serverless evaluation |

**References:**
- [AWS CodePipeline User Guide](https://docs.aws.amazon.com/codepipeline/latest/userguide/welcome.html)
- [Lambda Invoke Action](https://docs.aws.amazon.com/codepipeline/latest/userguide/action-reference-Lambda.html)
- [CodeBuild Buildspec Reference](https://docs.aws.amazon.com/codebuild/latest/userguide/build-spec-ref.html)
- [Langfuse External Evaluation Pipelines](https://langfuse.com/docs/scores/external-evaluation-pipelines)

## Pipeline Architecture

```
+---------------------------------------------------------------------------+
|                        AWS CodePipeline                                   |
+---------------------------------------------------------------------------+
|                                                                           |
|  +----------+    +-----------+    +-----------+    +----------+            |
|  |  Source  |--->|   Build   |--->|   Test    |--->| Approval |--->Deploy  |
|  |(CodeCommit)|  |(CodeBuild)|   | (Lambda)  |    | (Manual) |            |
|  +----------+    +-----------+    +-----------+    +----------+            |
|                                        |                                  |
|                                        v                                  |
|                              +-----------------+                          |
|                              |    Langfuse     |                          |
|                              |  (Self-Hosted)  |                          |
|                              |                 |                          |
|                              | - Fetch dataset |                          |
|                              | - Push scores   |                          |
|                              | - Quality gate  |                          |
|                              +-----------------+                          |
|                                                                           |
+---------------------------------------------------------------------------+
```

**Flow:**
1. Developer commits a prompt change to CodeCommit
2. CodeBuild installs dependencies and runs `evaluate_prompt.py`
3. The script fetches the evaluation dataset from Langfuse, runs each item through the agent, and pushes scores back
4. A quality gate checks if scores meet thresholds -- pipeline fails if they do not
5. Manual approval step before production deployment
6. Lambda function deploys the new prompt version to Langfuse with the `production` label

## Component 1: Evaluation Script

The evaluation script (`scripts/evaluate_prompt.py`) is the core of the pipeline. It can run locally during development or inside CodeBuild during CI.

Let us review the key parts and then run it locally.

In [ ]:
# ── Review the evaluation script ──────────────────────────────────────

eval_script_path = Path("scripts/evaluate_prompt.py")
if eval_script_path.exists():
    content = eval_script_path.read_text(encoding="utf-8")
    # Show the first 60 lines (docstring + config + helpers)
    lines = content.split("\n")
    print(f"File: {eval_script_path} ({len(lines)} lines)")
    print("=" * 70)
    for line in lines[:60]:
        print(line)
    print("...")
    print("=" * 70)
else:
    print(f"Script not found at {eval_script_path}")
    print("Expected location: 03-advanced-concepts/scripts/evaluate_prompt.py")

### Run the Evaluation Script Locally

You can run the evaluation script directly from this notebook. It will use the local dataset file as a fallback if Langfuse is not configured.

In [ ]:
# ── Run evaluation locally (inline, not subprocess) ────────────────────
# We import and run the core function to avoid subprocess complexity.

import importlib.util

spec = importlib.util.spec_from_file_location("evaluate_prompt", "scripts/evaluate_prompt.py")
eval_module = importlib.util.module_from_spec(spec)

try:
    spec.loader.exec_module(eval_module)
    # The module's main block only runs when __name__ == "__main__",
    # so we call the function directly:
    results = eval_module.run_evaluation()

    print(f"\n{'=' * 60}")
    print("LOCAL EVALUATION RESULTS")
    print(f"{'=' * 60}")
    print(f"  Total items:    {results['total']}")
    print(f"  Successful:     {results['successful']}")
    print(f"  Failed:         {results['failed']}")
    print(f"  Average score:  {results['avg_score']:.3f}")
    print(f"  Pass rate:      {results['pass_rate']:.1%}")
    print(f"{'=' * 60}")

    # Quality gate
    if results["avg_score"] >= 0.7 and results["pass_rate"] >= 0.8:
        print("\nQuality gate: PASSED")
    else:
        print("\nQuality gate: FAILED")

except Exception as e:
    print(f"Error running evaluation: {e}")
    print("This is expected if running without full AWS/Langfuse credentials.")

## Component 2: CodeBuild Buildspec

The buildspec tells CodeBuild how to run the evaluation. Key features:
- Secrets from AWS Secrets Manager (Langfuse credentials)
- Python 3.11 runtime
- Runs `evaluate_prompt.py` followed by `check_quality_gate.py`
- Generates a JSON report for CodeBuild reports

In [ ]:
# ── Display the buildspec ──────────────────────────────────────────────

buildspec = """
# buildspec.yml -- Place in your CodeCommit repository root
version: 0.2

env:
  secrets-manager:
    LANGFUSE_PUBLIC_KEY: "langfuse-pipeline-credentials:public_key"
    LANGFUSE_SECRET_KEY: "langfuse-pipeline-credentials:secret_key"
  variables:
    LANGFUSE_BASE_URL: "https://your-langfuse.internal.company.com"
    EVAL_DATASET_NAME: "customer-support-eval"
    EVAL_SCORE_THRESHOLD: "0.7"
    EVAL_PASS_RATE_THRESHOLD: "0.8"

phases:
  install:
    runtime-versions:
      python: 3.11
    commands:
      - pip install -q langfuse boto3

  pre_build:
    commands:
      - echo "Fetching evaluation dataset from Langfuse..."
      - echo "Dataset = $EVAL_DATASET_NAME"

  build:
    commands:
      - echo "Running prompt evaluation..."
      - python scripts/evaluate_prompt.py

  post_build:
    commands:
      - echo "Checking quality gate..."
      - python scripts/check_quality_gate.py

reports:
  prompt-eval-report:
    files:
      - "evaluation_results.json"
    file-format: "GENERICJSONFORMAT"

cache:
  paths:
    - "/root/.cache/pip/**/*"
"""

print(buildspec)

## Component 3: Lambda Function for CodePipeline

For the **Test** stage, a Lambda function can run evaluations and report success/failure back to CodePipeline using `PutJobSuccessResult` / `PutJobFailureResult`.

In [ ]:
# ── Lambda function for CodePipeline test action ───────────────────────
# This is a reference implementation -- deploy as a Lambda function.

lambda_code = '''
import json
import os
import boto3

codepipeline = boto3.client("codepipeline")
secrets_manager = boto3.client("secretsmanager")


def lambda_handler(event, context):
    """CodePipeline test action: run evaluation and report result."""
    job_id = event["CodePipeline.job"]["id"]

    try:
        # Retrieve Langfuse credentials from Secrets Manager
        secret_arn = os.environ["LANGFUSE_CREDENTIALS_SECRET"]
        secret_value = secrets_manager.get_secret_value(SecretId=secret_arn)
        creds = json.loads(secret_value["SecretString"])
        os.environ["LANGFUSE_PUBLIC_KEY"] = creds["public_key"]
        os.environ["LANGFUSE_SECRET_KEY"] = creds["secret_key"]

        # Parse user parameters from pipeline configuration
        user_params = (
            event["CodePipeline.job"]
            .get("data", {})
            .get("actionConfiguration", {})
            .get("configuration", {})
            .get("UserParameters", "{}")
        )
        params = json.loads(user_params) if user_params else {}
        threshold = float(params.get("threshold", os.environ.get("EVAL_SCORE_THRESHOLD", "0.7")))

        # Run evaluation (import your actual evaluation module)
        # from evaluate_prompt import run_evaluation
        # results = run_evaluation()
        results = {"avg_score": 0.85, "pass_rate": 1.0}  # Placeholder

        # Report result to CodePipeline
        if results["avg_score"] >= threshold:
            codepipeline.put_job_success_result(
                jobId=job_id,
                outputVariables={
                    "avg_score": str(results["avg_score"]),
                    "pass_rate": str(results["pass_rate"]),
                },
            )
            return {"status": "PASSED", "results": results}
        else:
            codepipeline.put_job_failure_result(
                jobId=job_id,
                failureDetails={
                    "type": "JobFailed",
                    "message": f"Quality gate failed: avg_score={results[\\"avg_score\\"]:.2f}",
                },
            )
            return {"status": "FAILED", "results": results}

    except Exception as e:
        codepipeline.put_job_failure_result(
            jobId=job_id,
            failureDetails={"type": "JobFailed", "message": str(e)},
        )
        raise
'''

print("Lambda function for CodePipeline test action:")
print("=" * 60)
print(lambda_code)
print("=" * 60)
print("\nKey points:")
print("  - Retrieves Langfuse credentials from Secrets Manager")
print("  - Calls put_job_success_result or put_job_failure_result")
print("  - Accepts threshold via UserParameters from pipeline config")

## Component 4: CloudFormation Template

The full pipeline is defined as infrastructure-as-code in `cloudformation/prompt-pipeline.yaml`. Let us review the key resources.

In [ ]:
# ── Review CloudFormation template ────────────────────────────────────

cfn_path = Path("cloudformation/prompt-pipeline.yaml")
if cfn_path.exists():
    cfn_content = cfn_path.read_text(encoding="utf-8")
    print(f"File: {cfn_path} ({len(cfn_content.splitlines())} lines)")
    print("=" * 70)

    # Show resources section summary
    print("\nResources defined in the template:\n")
    resources = [
        ("LangfuseCredentials", "AWS::SecretsManager::Secret", "Langfuse API credentials"),
        ("ArtifactBucket", "AWS::S3::Bucket", "Pipeline artifact storage"),
        ("EvaluationProject", "AWS::CodeBuild::Project", "Runs evaluate_prompt.py"),
        ("EvaluationLambda", "AWS::Lambda::Function", "CodePipeline test action"),
        ("PromptPipeline", "AWS::CodePipeline::Pipeline", "Source -> Build -> Test -> Approval -> Deploy"),
        ("CodeBuildRole", "AWS::IAM::Role", "Bedrock + Secrets Manager access"),
        ("LambdaRole", "AWS::IAM::Role", "CodePipeline + Bedrock access"),
        ("PipelineRole", "AWS::IAM::Role", "Pipeline orchestration"),
    ]

    for name, res_type, desc in resources:
        print(f"  {name:<25} {res_type:<40} {desc}")

    print(f"\n{'=' * 70}")
    print("\nPipeline stages:")
    print("  1. Source     - CodeCommit (triggers on push to main)")
    print("  2. Build      - CodeBuild (install deps, run evaluation)")
    print("  3. Approval   - Manual review before production deployment")
    print("  4. Deploy     - Lambda (promote prompt to production label)")
    print(f"\nDeploy with: aws cloudformation deploy --template-file {cfn_path} --stack-name prompt-pipeline --capabilities CAPABILITY_NAMED_IAM")
else:
    print(f"CloudFormation template not found at {cfn_path}")

## Quality Gate Script

The quality gate script (`scripts/check_quality_gate.py`) is a lightweight checker that reads the JSON results file and exits with non-zero status if thresholds are not met. It has no Langfuse or Bedrock dependencies, making it safe to run as a separate pipeline step.

In [ ]:
# ── Review the quality gate script ─────────────────────────────────────

gate_script_path = Path("scripts/check_quality_gate.py")
if gate_script_path.exists():
    content = gate_script_path.read_text(encoding="utf-8")
    lines = content.split("\n")
    print(f"File: {gate_script_path} ({len(lines)} lines)")
    print("=" * 70)
    for line in lines[:40]:
        print(line)
    print("...")
    print("=" * 70)
else:
    print(f"Quality gate script not found at {gate_script_path}")

## Alternative: GitHub Actions (Reference)

For teams using GitHub instead of CodeCommit, the same evaluation script works in a GitHub Actions workflow:

```yaml
# .github/workflows/prompt-eval.yml
name: Prompt Evaluation

on:
  pull_request:
    paths:
      - 'prompts/**'
      - 'agents/**'

jobs:
  evaluate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install langfuse boto3

      - name: Configure AWS credentials
        uses: aws-actions/configure-aws-credentials@v4
        with:
          aws-access-key-id: ${{ secrets.AWS_ACCESS_KEY_ID }}
          aws-secret-access-key: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
          aws-region: us-east-1

      - name: Run evaluation
        env:
          LANGFUSE_PUBLIC_KEY: ${{ secrets.LANGFUSE_PUBLIC_KEY }}
          LANGFUSE_SECRET_KEY: ${{ secrets.LANGFUSE_SECRET_KEY }}
          LANGFUSE_BASE_URL: ${{ secrets.LANGFUSE_BASE_URL }}
        run: python scripts/evaluate_prompt.py

      - name: Check quality gate
        run: python scripts/check_quality_gate.py
```

The key difference is where secrets are stored (GitHub Secrets vs AWS Secrets Manager) and how the pipeline is triggered (PR events vs CodeCommit push).

---

# Section 5: Summary and Production Checklist (5 min)

## Key Takeaways

| Area | What You Learned |
|------|------------------|
| **Prompt Versioning** | Store prompts in Langfuse, link to traces, analyze performance by version |
| **Systematic Evaluation** | Use Langfuse Datasets for repeatable test cases and experiments |
| **Automated Scoring** | LLM-as-Judge for scalable quality assessment with keyword + judge scoring |
| **CI/CD Integration** | AWS CodePipeline with Langfuse as quality gate, buildspec, Lambda action |

## Production Checklist

### Prompt Management
- [ ] Prompts stored in Langfuse (not just Git)
- [ ] Production label assigned to stable versions
- [ ] Traces linked to prompt versions for analytics

### Evaluation
- [ ] Dataset created with 20+ test cases
- [ ] Test cases cover: happy path, edge cases, failure modes
- [ ] Baseline metrics established for comparison

### Quality Scoring
- [ ] LLM-as-Judge evaluator configured
- [ ] Quality thresholds defined (e.g., avg_score >= 0.7)
- [ ] Score analytics dashboard reviewed regularly
- [ ] Alerts configured for score regressions

### CI/CD
- [ ] Pipeline triggers on prompt/agent changes
- [ ] Quality gate blocks deployment on failure
- [ ] Manual approval step for production deployments
- [ ] Rollback plan documented and tested

In [ ]:
# ── Final summary ────────────────────────────────────────────────────

print("=" * 70)
print("PRODUCTION PROMPT LIFECYCLE -- SUMMARY")
print("=" * 70)
print("""
Files created in this module:

  03-advanced-concepts/
  +-- 03-production-prompt-lifecycle.ipynb  (this notebook)
  +-- scripts/
  |   +-- evaluate_prompt.py               (CI/CD evaluation script)
  |   +-- check_quality_gate.py            (quality gate checker)
  +-- cloudformation/
  |   +-- prompt-pipeline.yaml             (CodePipeline template)
  +-- data/
      +-- customer-support-eval.json       (evaluation dataset)

Next Steps:

  1. Migrate existing prompts to Langfuse Prompt Management
  2. Build a dataset from production failures and edge cases
  3. Configure LLM-as-Judge evaluators for your quality criteria
  4. Deploy the CodePipeline (or GitHub Actions) for automated evaluation
  5. Monitor scores and iterate on prompts based on data
""")

if LANGFUSE_AVAILABLE:
    print(f"Langfuse dashboard: {langfuse_host}")
    print(f"  Prompts:  {langfuse_host}/prompts")
    print(f"  Datasets: {langfuse_host}/datasets")
    print(f"  Scores:   {langfuse_host}/scores")

print("=" * 70)

---

## Additional Resources

| Topic | Link |
|-------|------|
| Langfuse Prompt Management | https://langfuse.com/docs/prompts/get-started |
| Langfuse Datasets | https://langfuse.com/docs/datasets/overview |
| Langfuse Scores | https://langfuse.com/docs/scores/overview |
| Langfuse LLM-as-Judge | https://langfuse.com/docs/scores/model-based-evals |
| Langfuse External Pipelines | https://langfuse.com/docs/scores/external-evaluation-pipelines |
| Langfuse Self-Hosting | https://langfuse.com/docs/deployment/self-host |
| AWS CodePipeline | https://docs.aws.amazon.com/codepipeline/latest/userguide/welcome.html |
| CodePipeline Lambda Action | https://docs.aws.amazon.com/codepipeline/latest/userguide/action-reference-Lambda.html |
| CodeBuild Buildspec Reference | https://docs.aws.amazon.com/codebuild/latest/userguide/build-spec-ref.html |